# Can a longer context help Humanoid stay upright?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ccnets-team/causal-gpt-rl/blob/main/examples/can_longer_context_help_humanoid.ipynb)

One MuJoCo bundle drives a Humanoid for a thousand steps. This notebook runs it
seven times with the same weights and changes one thing between runs: how many
steps of the past the policy keeps — 16, 32, 64, 128, 256, 512, then 1000.

Nothing is retrained in between. Retention is a load-time argument, so the seven
runs load the same file and differ by one number.

The answer is not "more is better". It is a peak.

## Install

`mujoco` is pinned to `3.2.3`. A different simulator release is a different
measurement even with identical weights and seeds, so the numbers below are
defined on this one.

In [ ]:
%pip install -q "causal-gpt-rl[hub,mujoco]" "mujoco==3.2.3"

## Get the repository

The sweep reuses the rollout loop from
[`examples/deploy/reproduce.py`](https://github.com/ccnets-team/causal-gpt-rl/blob/main/examples/deploy/reproduce.py),
which is a file of this repository rather than part of the installed package. On
Colab, clone it; in a checkout, this cell does nothing.

In [ ]:
from pathlib import Path

if not Path("examples/deploy/reproduce.py").is_file():
    !git clone -q https://github.com/ccnets-team/causal-gpt-rl.git
    %cd causal-gpt-rl

## The knob

`kv_cache_max_len` is how many steps of the past a rollout keeps. It defaults to
the bundle's `context_length` — the window the policy was trained on, **32**
here — and that window is not a cap: the sweep runs the same policy with a
half of it, with it, and with up to thirty times it.

That is the entire difference between the seven runs:

```python
load_runner_from_hub(..., kv_cache_max_len=16)     # half of the window
load_runner_from_hub(..., kv_cache_max_len=32)     # the window it was trained on
load_runner_from_hub(..., kv_cache_max_len=128)    # four times past it
load_runner_from_hub(..., kv_cache_max_len=1000)   # the whole episode
```

In [ ]:
import torch

from examples.deploy.reproduce import installed_versions, print_stack_report

repo_id = "ccnets/causal-gpt-rl"
subfolder = "humanoid-v5"
env_id = "Humanoid-v5"

CONTEXT_VALUES = [16, 32, 64, 128, 256, 512, 1000]  # the bundle's own is 32

# One seed per episode. The episode count is also the width of the batch the
# policy runs as, and that width is part of the measurement — lowering it for a
# quicker look changes the numbers, it does not just shorten the run.
episodes = 50
max_steps = 1000
seeds = list(range(episodes))
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"{repo_id}/{subfolder} on {env_id}  ({device})\n")
print_stack_report(installed_versions())

## Run the sweep

Each setting gets a fresh policy and a fresh set of environments, and every one
of them runs the same seeds. The seven run one after another rather than at
once, so the timing is what a single user would see.

Expect roughly four minutes on a recent GPU. On CPU it is much longer — on
Colab, check that the runtime has a GPU attached before starting.

In [ ]:
import gymnasium as gym
import numpy as np

from causal_gpt_rl.inference import load_runner_from_hub
from examples.deploy.reproduce import run_seed_batch

results = []

for context in CONTEXT_VALUES:
    policy = load_runner_from_hub(
        repo_id=repo_id,
        subfolder=subfolder,
        device=device,
        num_envs=episodes,
        kv_cache_max_len=context,
    )
    envs = gym.vector.SyncVectorEnv(
        [lambda eid=env_id: gym.make(eid) for _ in seeds],
        autoreset_mode=gym.vector.AutoresetMode.SAME_STEP,
    )
    try:
        returns, lengths, ends = run_seed_batch(
            envs, policy, seeds, max_steps, per_seed=False
        )
    finally:
        envs.close()

    # `terminated` is the Humanoid falling over; `truncated` is the thousand-step
    # limit arriving with it still upright. Only the first one is a failure.
    fell = [int(seeds[row]) for row in np.flatnonzero(ends["terminated"])]
    results.append({
        "context": context,
        "trained": policy.context_length,   # the window the bundle was trained on
        "returns": returns,
        "full": episodes - len(fell),
        "fell": fell,
    })
    print(f"context={context:<5} mean={returns.mean():8.2f}  "
          f"worst={returns.min():8.2f}  "
          f"{results[-1]['full']}/{episodes} full episodes", flush=True)

## The table

The mean is the least interesting column. What moves is the bottom of the range:
the **worst episode**, how many rollouts stayed upright for the full thousand
steps instead of falling before the limit arrived, and how tightly the returns
cluster.

In [ ]:
header = (f"{'context':>9}   {'return':>12}   {'std':>10}   "
          f"{'worst episode':>15}   {'full episodes':>15}")
rule = "-" * len(header)

print(header)
print(rule)
for r in results:
    returns = r["returns"]
    full = f"{r['full']} / {episodes}"
    mark = "   <- trained window" if r["context"] == r["trained"] else ""
    print(f"{r['context']:>9}   {returns.mean():12.2f}   {returns.std():10.2f}   "
          f"{returns.min():15.2f}   {full:>15}{mark}")
print(rule)

print("\nseeds where the Humanoid fell:")
for r in results:
    print(f"  context={r['context']:<5} {r['fell'] or 'none'}")

## How Much Context Does a Humanoid Need to Walk?

### What we measured

| context | return | std | worst episode | full episodes |
|:---|---:|---:|---:|---:|
| 16 | 7406.59 | 2024.57 | 496.91 | 45 / 50 |
| 32 &nbsp;— *trained window* | 7572.63 | 1437.13 | 793.72 | 43 / 50 |
| 64 | 7762.28 | 1133.43 | 1944.22 | 47 / 50 |
| **128** &nbsp;— *best result* | **8037.26** | **129.16** | **7334.43** | **48 / 50** |
| 256 | 7459.69 | 1731.94 | 1768.51 | 43 / 50 |
| 512 | 7364.14 | 1822.50 | 1768.51 | 42 / 50 |
| 1000 | 7389.61 | 1804.21 | 1768.51 | 41 / 50 |

Notebooks here are committed without outputs. These results were produced on our
machine using the Hugging Face bundle with MuJoCo 3.2.3, Gymnasium 1.2.3, and
torch 2.8.0.

The means span 9% and, apart from `context=128`, they are not ordered by
anything: a fifty-seed draw of this protocol does not separate 7364 from 7762.

The spread does separate. Read the `std` column down the table:

```
context     16      32      64     128     256     512    1000
    std   2025    1437    1133     129    1732    1822    1804
```

One order of magnitude, at one setting, approached monotonically from the left
and left behind immediately on the right. The worst episode says the same thing
— 497, 794, 1944, **7334**, 1769, 1769, 1769 — the floor is lifted at
`context=128` and nowhere else.

**A short memory does not make the policy walk worse. It makes it fall over
sometimes. So does a long one.** `context=128` is four times the window this
policy was trained on; `context=256` is eight times, and it is worse than the
trained window itself.

Retention was close to free in wall time: every run took roughly half a minute,
and the differences between them did not order by `context`.

## What this shows, and what it does not

**One bundle, one environment.** This Humanoid has an optimum at four times its
trained window. Across the published bundles retention helps some and hurts
others, so this is a measurement to repeat in your own environment, not a number
to copy.

**Longer is not automatically better.** Past `context=128` the policy returns to
the scores and the variance of the short settings. Retention buys stability up
to a point and then stops.

**The episode bounds retention.** A cache only holds the steps that have
actually run, so with `max_steps=1000` any value above 1000 behaves exactly like
1000 — the `context=1000` run never reaches its cap, it simply keeps everything.

**Fifty seeds is one draw.** Which seeds fall is sensitive: two runs that differ
only in floating-point ordering can trade several episodes. Differences of a few
hundred return do not survive a reshuffle. The collapse in spread at
`context=128` does.

**The batch width is part of the measurement.** Fifty episodes run as one
fifty-row batch, and a batch of one reduces floating point in a different order
— in a closed loop that difference compounds. Changing `episodes` above changes
both the sample and the batch, so a shorter run is a different measurement
rather than a rougher one. A row that finishes early keeps stepping under the
vector env's auto-reset, but it is out of the scoring and its context is left
alone, so it never disturbs the rows still being scored.

To measure a bundle under the published protocol, or to try another environment:

```
python -m examples.deploy.reproduce --env-id Humanoid-v5 --kv-cache-max-len 128
```

More on what retention is, and why the trained window is not a ceiling:
[Rollout History](https://github.com/ccnets-team/causal-gpt-rl#rollout-history).